# Relatedness / Duplicate Filtering — European-American Cohort (GSE148812)

**Purpose:** Identify and remove duplicate/closely-related samples that violate
the independent-samples assumption underlying DoubleML and downstream tests.
Discovered via genomic inflation factor (λ=1.98 at 10 PCs, worsening with more
PCs — signature of relatedness, not population structure).

**Method:** Pairwise genotype correlation on a random SNP subset → build a graph
of related samples → keep one representative per connected cluster (lowest
missingness), drop the rest.

**Threshold used:** correlation > 0.5 only (near-certain duplicates + close
relatives). The 0.3–0.5 band is NOT filtered here — flagged as a documented
limitation, since it's harder to distinguish from admixture-driven correlation,
especially in AA.

**Inputs:** `checkpoint7_snp_encoded_012.csv`, `checkpoint2_metadata_sample_filtered.csv`

**Outputs:** `checkpoint2b_metadata_relatedness_filtered.csv`,
`qc_log_dropped_relatedness.csv` (which samples dropped and why)

In [3]:
import pandas as pd
import numpy as np
import re
import os
import gc

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

encoded_df = pd.read_csv(os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv"))
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_cols = encoded_df.columns[1:]
sample_ids = sample_cols.tolist()
X_snp_first = encoded_df[sample_cols].to_numpy(dtype=np.int8)
del encoded_df
gc.collect()

manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_probes = set(manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "IlmnID"])
sex_linked_core_names = {strip_address_suffix(p) for p in sex_linked_probes}
our_probe_core_names = {strip_address_suffix(p): p for p in probe_id_array}
to_exclude = {our_probe_core_names[c] for c in sex_linked_core_names if c in our_probe_core_names}
del manifest_df
gc.collect()

keep_mask = ~np.isin(probe_id_array, list(to_exclude))
X_auto_int = X_snp_first[keep_mask]
del X_snp_first
gc.collect()

X_f32 = X_auto_int.astype(np.float32)
p_ea = X_f32.mean(axis=1) / 2
denom_ea = np.sqrt(2 * p_ea * (1 - p_ea))
valid_mask = denom_ea > 1e-8
del X_f32
gc.collect()

valid_indices = np.where(valid_mask)[0]
np.random.seed(0)
chosen_idx = np.random.RandomState(0).choice(valid_indices, 5000, replace=False)

X_subset_int = X_auto_int[chosen_idx].astype(np.float64)
del X_auto_int
gc.collect()

p_subset = p_ea[chosen_idx]
denom_subset = denom_ea[chosen_idx]
X_subset_std = ((X_subset_int - 2 * p_subset[:, None]) / denom_subset[:, None]).T
del X_subset_int
gc.collect()

sample_corr = np.corrcoef(X_subset_std)
del X_subset_std
gc.collect()
np.fill_diagonal(sample_corr, 0)

print("Correlation matrix rebuilt, shape:", sample_corr.shape)
print("Sample count:", len(sample_ids))

Correlation matrix rebuilt, shape: (1595, 1595)
Sample count: 1595


In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

# reuse: sample_ids, sample_corr_ea (or recompute if not in memory)
# assuming sample_corr matrix and sample_ids from the EA relatedness check are still in this session

threshold = 0.5
iu = np.triu_indices_from(sample_corr, k=1)  # use EA's correlation matrix from earlier
rows, cols = iu
strong = sample_corr[iu] > threshold

pairs = [(sample_ids[rows[k]], sample_ids[cols[k]], sample_corr[iu][k])
         for k in range(len(strong)) if strong[k]]

print(f"Pairs above threshold {threshold}: {len(pairs)}")

# build graph, find connected clusters (handles chains: A-B-C all related)
G = nx.Graph()
G.add_nodes_from(sample_ids)
for a, b, r in pairs:
    G.add_edge(a, b, weight=r)

clusters = [c for c in nx.connected_components(G) if len(c) > 1]
print(f"Number of related clusters: {len(clusters)}")
print(f"Total samples involved: {sum(len(c) for c in clusters)}")

# tie-break: keep the sample with the LOWEST sample_id (simple, deterministic, documented)
# (a missingness-based tie-break would be better but requires re-deriving per-sample
#  missingness from checkpoint0/checkpoint3 — noted as a possible refinement)
samples_to_drop = []
for cluster in clusters:
    sorted_cluster = sorted(cluster)
    keep = sorted_cluster[0]
    drop = sorted_cluster[1:]
    samples_to_drop.extend(drop)

print(f"\nSamples to drop: {len(samples_to_drop)}")
print(f"Samples remaining: {len(sample_ids) - len(samples_to_drop)}")

pd.Series(samples_to_drop, name="dropped_sample_id_relatedness").to_csv(
    os.path.join(out_dir, "qc_log_dropped_relatedness.csv"), index=False
)
print("Saved drop log.")

Pairs above threshold 0.5: 208
Number of related clusters: 88
Total samples involved: 223

Samples to drop: 135
Samples remaining: 1460
Saved drop log.


In [5]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"

samples_to_drop_set = set(samples_to_drop)

meta_df = pd.read_csv(os.path.join(out_dir, "checkpoint2_metadata_sample_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)

meta_df_relatedness_filtered = meta_df[~meta_df["sample_id"].isin(samples_to_drop_set)].copy()
print("Metadata shape before:", meta_df.shape)
print("Metadata shape after relatedness filter:", meta_df_relatedness_filtered.shape)

meta_df_relatedness_filtered.to_csv(
    os.path.join(out_dir, "checkpoint2b_metadata_relatedness_filtered.csv"), index=False
)
print("Saved.")

Metadata shape before: (1595, 9)
Metadata shape after relatedness filter: (1460, 9)
Saved.


In [6]:
import pandas as pd
import os

out_dir = r"C:\Users\user\Downloads\GSE148812_clean"
geno_path = os.path.join(out_dir, "checkpoint7_snp_encoded_012.csv")

chunksize = 20000
out_path = os.path.join(out_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv")
first_chunk = True

reader = pd.read_csv(geno_path, chunksize=chunksize)
for chunk in reader:
    cols_to_keep = [c for c in chunk.columns if c == "probe_id" or c not in samples_to_drop_set]
    chunk = chunk[cols_to_keep]
    chunk.to_csv(out_path, mode="w" if first_chunk else "a", header=first_chunk, index=False)
    first_chunk = False

print("Saved relatedness-filtered genotype file:", out_path)

# quick verification
check_df = pd.read_csv(out_path, nrows=0)
print("Sample columns remaining:", len(check_df.columns) - 1, "(expect 1460)")

Saved relatedness-filtered genotype file: C:\Users\user\Downloads\GSE148812_clean\checkpoint7b_snp_encoded_012_relatedness_filtered.csv
Sample columns remaining: 1460 (expect 1460)
